# Phase 9a: Llama 3.2 1B cross-family replication

Requires A100 GPU runtime: Runtime > Change runtime type > A100 GPU.

**Setup:** Add your HuggingFace token as a Colab secret named `HF_TOKEN`
(key icon in left sidebar), or paste it directly in the auth cell below.

In [ ]:
!git clone https://github.com/tmcarmichael/nn-observability.git
%cd nn-observability

In [ ]:
!pip install uv -q
!uv sync --extra transformer -q

In [ ]:
import os
os.environ.pop("MPLBACKEND", None)

# Try Colab secrets first, fall back to manual entry
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except (ImportError, userdata.SecretNotFoundError):
    os.environ["HF_TOKEN"] = ""  # paste your token here if not using secrets
    if not os.environ["HF_TOKEN"]:
        print("WARNING: Set HF_TOKEN above or add it as a Colab secret")

In [ ]:
# Note: phase9a uses run_cross_family which loads the model internally.
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print('bfloat16 + sdpa enabled in run_cross_family via AutoModel')

In [ ]:
import json
d = json.load(open('results/cross_family.json'))
for phase, data in d.items():
    for model, v in data['models'].items():
        pc = v['partial_corr']['mean']
        oc = v['output_controlled']['mean']
        sa = v['seed_agreement']['mean']
        pk = v['peak_layer']
        nl = v['n_layers']
        print(f"{model}: L{pk} ({pk/nl:.0%}) pcorr={pc:+.4f} ctrl={oc:+.4f} agree={sa:+.4f}")
        print(f"  baselines: {v['baselines']}")

In [ ]:
from google.colab import files
files.download('results/cross_family.json')